# Train Lora + LLM Judge + RAG Evaluation

**Nguyen tac cua notebook nay:**
- **1 co `USE_RAG` DUY NHAT** dung chung cho Buoc 5 (run_evaluation) va
  Buoc 6 (LLM Judge) -- de dam bao evaluate LUON khop dieu kien luc train
  (train khong RAG thi evaluate cung khong RAG, va nguoc lai).
- **Chat (Buoc 7)** la doc lap -- tu chon bat/tat RAG rieng cho tung cau
  hoi bang lenh `/rag on` / `/rag off`, KHONG anh huong den co USE_RAG
  chung o tren.

**QUAN TRONG -- doc truoc khi chay:**
1. Runtime -> Disconnect and delete runtime (don sach session cu).
2. Runtime -> Change runtime type -> GPU (T4).
3. Chay tuan tu tu tren xuong, KHONG bo qua cell nao.
4. Sau Buoc 3 (cai dat), co cell yeu cau RESTART SESSION -- bat buoc.
5. Buoc 5 (diagnostic) neu co dong "[FAIL]" thi DUNG LAI xu ly truoc.


# Phần 1: Chuẩn bị cài đặt môi trường

## Buoc 0: Kiem tra GPU

In [1]:
!nvidia-smi


Wed Jul 22 09:37:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("\n[FAIL] KHONG CO GPU. Doi Runtime type -> GPU (T4), roi Disconnect and delete runtime.")
else:
    print("[OK] GPU san sang.")


torch.cuda.is_available(): True
[OK] GPU san sang.


## Buoc 1: Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Buoc 2: Clone project SACH + PATCH ngay (truoc khi cai dat)

Patch `trust_remote_code=True` cho nomic-embed-text phai lam NGAY SAU KHI
CLONE, truoc ca buoc cai dat/diagnostic -- neu de sau se bi FAIL o buoc
kiem tra rag_bridge.

In [4]:
import shutil, os

PROJECT_DIR = "/content/MockProject_062026_NhomAI"

if os.path.exists(PROJECT_DIR):
    print(f"Xoa ban clone cu tai {PROJECT_DIR} ...")
    shutil.rmtree(PROJECT_DIR)

for stray in ["/content/MockProject_062026_NhomAI"]:
    if os.path.exists(stray):
        print(f"Xoa ban clone LAC tai {stray} ...")
        shutil.rmtree(stray)

print("Da don sach. Bat dau clone moi...")


Da don sach. Bat dau clone moi...


In [5]:
!git clone --branch tuanphat --single-branch https://github.com/internvietridao/MockProject_062026_NhomAI.git /content/MockProject_062026_NhomAI

%cd /content/MockProject_062026_NhomAI/Training

!git clone --branch han --single-branch https://github.com/internvietridao/MockProject_062026_NhomAI.git _RAG_


Cloning into '/content/MockProject_062026_NhomAI'...
remote: Enumerating objects: 11679, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 11679 (delta 58), reused 61 (delta 35), pack-reused 11579 (from 2)
Receiving objects: 100% (11679/11679), 178.44 MiB | 17.97 MiB/s, done.
Resolving deltas: 100% (6998/6998), done.
Updating files: 100% (11422/11422), done.
/content/MockProject_062026_NhomAI/Training
Cloning into '_RAG_'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 222 (delta 3), reused 2 (delta 2), pack-reused 206 (from 1)
Receiving objects: 100% (222/222), 177.11 MiB | 18.59 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Updating files: 100% (116/116), done.


In [6]:
import os
assert os.getcwd() == "/content/MockProject_062026_NhomAI/Training", f"Sai working dir: {os.getcwd()}"
assert os.path.exists("pipeline/evaluate.py"), "Khong tim thay pipeline/evaluate.py -- clone loi"
assert os.path.exists("_RAG_/Embbeding_RAG"), "Khong tim thay _RAG_/Embbeding_RAG -- clone loi"
print("[OK] Clone dung vi tri, cau truc thu muc hop le.")


[OK] Clone dung vi tri, cau truc thu muc hop le.


### ---------------------------- *Just using this when using nomic embbeding!* :----------------------------

In [ ]:
# PATCH: nomic-embed-text-v1.5 can trust_remote_code=True (kien truc custom)
NOMIC_DIR = "/content/MockProject_062026_NhomAI/Training/_RAG_/Embbeding_RAG/nomic-embed-text-v1.5"

!sed -i 's/model_name=embedding_model_name$/model_name=embedding_model_name, trust_remote_code=True/' \
  "{NOMIC_DIR}/test_vector_db/retrieval.py"
!sed -i 's/model_name=EMBEDDING_MODEL_NAME$/model_name=EMBEDDING_MODEL_NAME, trust_remote_code=True/' \
  "{NOMIC_DIR}/embedding_storage.py"

# Xac nhan patch da vao dung file
!grep -n "trust_remote_code" "{NOMIC_DIR}/test_vector_db/retrieval.py" || echo "[FAIL] Patch chua vao retrieval.py"

### --------------------------------------------------------------------------------------------------------

## Buoc 3: Cai dat package (1 LAN DUY NHAT)

In [7]:
%cd /content/MockProject_062026_NhomAI/Training

!pip install --upgrade pip -q
!pip install \
    -r requirements_ver1.txt \
    -r _RAG_/Embbeding_RAG/requirements.txt \
    -q
!pip install -U "opentelemetry-api==1.44.0" "opentelemetry-sdk==1.44.0" -q

/content/MockProject_062026_NhomAI/Training
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 76.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.2.38 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which i

## >>> RESTART SESSION <<<

Runtime -> Restart session. Sau khi restart

In [3]:
print("Nho: Runtime -> Restart session, chay tiep Buoc 4.")

Nho: Runtime -> Restart session, chay tiep Buoc 4.


## Buoc 4: DAT CO USE_RAG CHUNG (dung cho ca Buoc 5 va Buoc 6)

Doi gia tri True/False O DAY DUY NHAT -- dam bao evaluate luon khop dieu
kien luc train (vd: train KHONG dung RAG -> de False; neu ban co train
lai VOI RAG thi doi thanh True).

In [4]:
import os
os.chdir("/content/MockProject_062026_NhomAI/Training")

import sys
sys.path.insert(0, "/content/MockProject_062026_NhomAI/Training")

# ============================================================
# >>> DOI GIA TRI NAY CHO KHOP VOI LUC TRAIN <<<
TRAIN_EVAL_USE_RAG = True   # True neu model duoc train/muon eval CO RAG
# ============================================================

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["MEDQUAD_RAG_DIR"] = "/content/MockProject_062026_NhomAI/Training/_RAG_/Embbeding_RAG/all-MiniLM-L6-v2"
os.environ["MEDQUAD_USE_RAG"] = "1" if TRAIN_EVAL_USE_RAG else "0"
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["CHROMA_TELEMETRY_IMPL"] = "none"

print(f"[OK] Da set MEDQUAD_USE_RAG = {os.environ['MEDQUAD_USE_RAG']} (dung cho Buoc 5 + Buoc 6)")


[OK] Da set MEDQUAD_USE_RAG = 1 (dung cho Buoc 5 + Buoc 6)


## Buoc 5: DIAGNOSTIC -- kiem tra du dieu kien truoc khi chay that

In [5]:
import torch
print(("[OK]" if torch.cuda.is_available() else "[FAIL]"), "GPU:", torch.cuda.is_available())


[OK] GPU: True


In [6]:
try:
    import opentelemetry.sdk.environment_variables as ev
    _ = ev.OTEL_LOGRECORD_ATTRIBUTE_COUNT_LIMIT
    print("[OK] opentelemetry OK. File:", ev.__file__)
except Exception as e:
    print("[FAIL] opentelemetry loi:", e)


[OK] opentelemetry OK. File: /usr/local/lib/python3.12/dist-packages/opentelemetry/sdk/environment_variables/__init__.py


In [7]:
try:
    import chromadb
    print("[OK] chromadb OK, version:", chromadb.__version__)
except Exception as e:
    print("[FAIL] chromadb loi:", e)


[OK] chromadb OK, version: 1.5.9


In [8]:
try:
    from ragas import evaluate as ragas_evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
    print("[OK] ragas OK.")
except Exception as e:
    print("[FAIL] ragas loi:", e)


[OK] ragas OK.


In [9]:
try:
    from src.config import USE_RAG, ADAPTER_DIR, PREDICTIONS_CSV
    print("[OK] src.config OK. USE_RAG =", USE_RAG)
    print("ADAPTER_DIR:", ADAPTER_DIR, "| ton tai:", ADAPTER_DIR.exists())
    print("PREDICTIONS_CSV:", PREDICTIONS_CSV)
except Exception as e:
    print("[FAIL] src.config loi:", e)


[OK] src.config OK. USE_RAG = True
ADAPTER_DIR: /content/MockProject_062026_NhomAI/Training/output/output_model | ton tai: True
PREDICTIONS_CSV: /content/MockProject_062026_NhomAI/Training/output/evaluation_results.csv


In [10]:
from src.config import USE_RAG
import os

if USE_RAG:
    rag_dir = os.environ["MEDQUAD_RAG_DIR"]
    chroma_path = os.path.join(rag_dir, "chroma_db", "chroma.sqlite3")
    if os.path.exists(chroma_path):
        print("[OK] chroma_db ton tai tai:", chroma_path)
    else:
        print(f"[FAIL] KHONG tim thay {chroma_path} -- can build chroma_db truoc.")
else:
    print("[SKIP] USE_RAG=False -- khong can check chroma_db.")


[OK] chroma_db ton tai tai: /content/MockProject_062026_NhomAI/Training/_RAG_/Embbeding_RAG/all-MiniLM-L6-v2/chroma_db/chroma.sqlite3


In [11]:
from src.config import USE_RAG

if USE_RAG:
    try:
        from src.rag_bridge import get_context_with_similarity
        r = get_context_with_similarity("What are the symptoms of diabetes?", top_k=2)
        if r["raw_contexts"]:
            print(f"[OK] rag_bridge retrieve OK -- {len(r['raw_contexts'])} chunks, rag_used={r['rag_used']}")
            print("Preview:", r["raw_contexts"][0][:200])
        else:
            print("[FAIL] rag_bridge chay khong loi nhung RONG -- kiem tra lai chroma_db co du lieu chua.")
    except Exception as e:
        print("[FAIL] rag_bridge loi:", type(e).__name__, "-", e)
else:
    print("[SKIP] USE_RAG=False -- khong can test rag_bridge.")


[rag_bridge] Đã load retrieval module từ: /content/MockProject_062026_NhomAI/Training/_RAG_/Embbeding_RAG/all-MiniLM-L6-v2
[rag_bridge] RETRIEVAL_MODE = bm25 | SIMILARITY_THRESHOLD = 0.7
[OK] rag_bridge retrieve OK -- 2 chunks, rag_used=True
Preview: Abolish symptoms of diabetes.

Correct hyperglycaemia, glycosuria.

Prevent and manage complications.


In [12]:
from src.config import ADAPTER_DIR
if ADAPTER_DIR.exists() and any(ADAPTER_DIR.iterdir()):
    print("[OK] Tim thay model da train tai:", ADAPTER_DIR)
else:
    print(f"[FAIL] Chua co model tai {ADAPTER_DIR} -- can train truoc hoac tai model len dung vi tri nay.")


[OK] Tim thay model da train tai: /content/MockProject_062026_NhomAI/Training/output/output_model


In [13]:
print("=" * 82)
print("Neu TAT CA cac CHECK deu [OK] (hoac [SKIP] khi USE_RAG=False) -> chay tiep Buoc 6.")
print("=" * 82)


Neu TAT CA cac CHECK deu [OK] (hoac [SKIP] khi USE_RAG=False) -> chay tiep Buoc 6.


# Phần 2: Train

## Bước 1: Build train.jsonl từ final_train_dataset.json

In [ ]:
from pipeline import build_train_dataset
build_train_dataset.main()

## Bước 2: Train Lora

### Bước 2.1: Training (Nếu đã có model -> Bỏ qua)

In [ ]:
from pipeline import train
train.main()

### Bước 2.2: Nếu đã có model

In [14]:
from pipeline import run_evaluation
#run_evaluation.main()   # full tap test, ghi đè từ đầu
run_evaluation.main(resume=True)  # dùng nếu bị ngắt giữa chừng


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Đang tải tập test (thô)...
Đang load model đã train từ /content/MockProject_062026_NhomAI/Training/output/output_model...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Load xong.
Số câu hỏi test: 5577 | Sẽ chạy: 5577
USE_RAG=True -> sẽ retrieve context THẬT + lọc theo % tương đồng (qua rag_bridge).
[EVAL] Resume: đã có 3385 mẫu trong /content/MockProject_062026_NhomAI/Training/output/evaluation_results.csv, sẽ bỏ qua các câu này và chỉ generate phần còn thiếu.
[EVAL] Mục tiêu: 5577 mẫu.
[EVAL] Đã inference 3390/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 3395/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 3400/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 3405/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 3410/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 3415/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 3420/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 3425/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 3430/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 3435/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 3440/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 3445/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 3450/5577 mẫu... (đã lưu CSV)
[EVAL] Đã inference 3455/5

## Buoc 3: LLM Judge (RAGAs)

In [17]:
import getpass
import importlib
os.environ["MEDQUAD_JUDGE_API_KEY"] = getpass.getpass("Nhập API Key")

import src.config
importlib.reload(src.config)

from pipeline import evaluate
importlib.reload(evaluate)
evaluate.main()

Nhập API Key··········
Kết quả đánh giá sẽ được lưu (append theo batch) vào: /content/drive/MyDrive/medquad_eval/ragas_scores.csv
Kết quả đánh giá sẽ được lưu (append theo batch) vào: /content/drive/MyDrive/medquad_eval/ragas_scores.csv
Đang đọc CSV dự đoán (đã sinh sẵn từ bước train, gồm ROUGE/BLEU)...
Đã đọc cột 'contexts' từ CSV: 5495/5495 mẫu có context thật (RAG) -> faithfulness/context_precision/context_recall có thể dùng được.
Số mẫu: 5495
Đang khởi tạo model giám khảo (tách biệt model vừa train)...
Bật JSON mode cho giám khảo (giảm lỗi parse với model nhỏ). Nếu thấy TOÀN BỘ câu bị lỗi/NaN sau khi bật, set MEDQUAD_JUDGE_JSON_MODE=0 và chạy lại để tắt JSON mode.
Gọi model giám khảo qua API: llama-3.1-8b-instant (https://api.groq.com/openai/v1)


/content/MockProject_062026_NhomAI/Training/pipeline/evaluate.py:132: LangChainBetaWarning: Introduced in 0.2.24. API subject to change.
  rate_limiter = InMemoryRateLimiter(


Đang load embedding model (cho Answer Relevance)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

ERROR:judge_debug:Lỗi gọi API giám khảo: BadRequestError: Error code: 400 - {'error': {'message': "Failed to generate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': '{\n  "text": "The output string did satisfy the constraints given in the prompt. Fix the output string and return it.\\nPlease return the output in a JSON format that complies with the following schema as specified in JSON Schema:\\n{\'properties\': {\'text\': {\'title\': \'Text\', \'type\': \'string\'}}, \'required\': [\'text\'], \'title\': \'StringIO\', \'type\': \'object\'}\\n\\n-----------------------------\\n\\nNow perform the same with the following input\\ninput: {\\n    \\"output_string\\": \\"{\\\\n  \\\\\\"sentences\\\\\\": [\\\\n      {\\\\n         \\\\\\"sentence_index\\\\\\": 0,\\\\n         \\\\\\"simpler_statements\\\\\\": [\\\\n            \\\\\\"The genetic causes of chronic granulomatous he

Đã lưu batch 10 vào /content/drive/MyDrive/medquad_eval/ragas_scores.csv
--- Batch 11 (5 câu, 55/5495) ---


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Buoc 4: CHAT -- doc lap, tu chon bat/tat RAG rieng (khong lien quan co USE_RAG o Buoc 4)

Trong luc chat, go:
- `/rag on`  -> bat RAG cho cac cau hoi tiep theo
- `/rag off` -> tat RAG (Q&A thuan)
- `exit` hoac `quit` -> thoat

Neu USE_RAG cua ban chua bat (Buoc 4 dang False) ma muon thu RAG o day,
can dam bao da co chroma_db + da patch trust_remote_code (Buoc 2).

In [18]:
from pipeline import chat
chat.chat_loop()


Đang load model...
Load model xong.

CHATBOT SẴN SÀNG (mặc định USE_RAG=True)
Lệnh: '/rag on' bật RAG | '/rag off' tắt RAG | 'exit'/'quit' thoát

Câu hỏi của bạn: what are the symptoms of heart disease

--- RAG cho câu hỏi này (mode=bm25) ---
  [1] Tương đối (so với tốt nhất): 100.0% | Heart failure occurs when the heart is unable to supply output that is sufficient for the metabolic needs of the tissues, in face of adequate venous r...
  [2] Tương đối (so với tốt nhất): 98.1% | The following general principles should guide the management of congenital heart disease:  Parents should be counselled on what can and what cannot be...
  [3] Tương đối (so với tốt nhất): 90.8% | 3. Cardiovascular Diseases  These are the diseases and disorders of the heart and blood vessels. They include rheumatic heart disease, coronary heart ...
  -> DÙNG RAG: 3 context đạt ngưỡng, đưa vào prompt.
--------------------------------------------------

TRẢ LỜI: Heart disease can cause shortness of breath, fatigu